# W08 · Application 2 — Neural Texture Compression / 神經材質壓縮

**English.** A PBR material is a 9-channel signal (albedo RGB, normal XY,
roughness, metalness, AO). We fit it with an NTC-style grid baseline and with
PEPS variants, then compare per-material PSNR — reproducing the structure of
paper Table 2. MetalPlates013 is exactly NVIDIA RTXNTC's demo set, giving a
direct point of comparison. PEPS's advantage is largest on high-frequency
metals and smallest on low-frequency wood — we show this honestly.

**繁體中文.** PBR 材質是 9 通道訊號(albedo RGB、normal XY、roughness、
metalness、AO)。用 NTC 風格 grid 基線與 PEPS 變體擬合,比較逐材質 PSNR,
重現論文 Table 2 結構。MetalPlates013 正是 NVIDIA RTXNTC 的示範材質,可直接
對照。PEPS 優勢在高頻金屬最大、低頻木頭最小 —— 誠實呈現。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, numpy as np, matplotlib.pyplot as plt
from peps.train import auto_device, fit, TrainConfig, render_full
from peps.metrics import psnr
from apps.texture.data import load_pbr_bundle, bundle_to_coords_targets, find_bundle, CHANNEL_LAYOUT
from apps.texture.build import build_ntc_baseline, build_grid_peps_texture
device = auto_device(); print('device', device, '| bundle channels', sum(c for _,c in CHANNEL_LAYOUT))

device cuda | bundle channels 9


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Materials spanning the frequency spectrum / 涵蓋頻率光譜的材質

In [2]:
sets = ['MetalPlates013', 'Metal032', 'Planks020', 'Rock023']
notes = {'MetalPlates013':'high-freq metal (NVIDIA demo)', 'Metal032':'metal',
         'Planks020':'low-freq wood', 'Rock023':'mid-freq noisy'}
bundles = {}
for s in sets:
    b = load_pbr_bundle(find_bundle(s), size=512)
    bundles[s] = b
    print(f'{s:16s} {b.shape}  ({notes[s]})')

MetalPlates013   torch.Size([512, 512, 9])  (high-freq metal (NVIDIA demo))


Metal032         torch.Size([512, 512, 9])  (metal)


Planks020        torch.Size([512, 512, 9])  (low-freq wood)


Rock023          torch.Size([512, 512, 9])  (mid-freq noisy)


## 2. Train NTC vs Grid-PEPS vs NTC_PEPS per material / 逐材質訓練三方法
Matched grid resolution/feature dim; PEPS samples the shared grid at 2L+1
Lissajous points. 相同 grid 解析度/特徵維度;PEPS 在 2L+1 個點取樣共享 grid。

In [3]:
methods = {
  'ntc':       lambda: build_ntc_baseline(resolution=256, feature_dim=8),
  'grid_peps': lambda: build_grid_peps_texture(256, 8, 6, 'concat'),
  'ntc_peps':  lambda: build_grid_peps_texture(256, 8, 6, 'pink'),
}
table = {m: {} for m in methods}
recon = {}  # keep MetalPlates013 recons for the side-by-side
for s in sets:
    b = bundles[s]
    coords, targets, (H, W) = bundle_to_coords_targets(b)
    for m, builder in methods.items():
        model, pc = builder()
        fit(model, coords, targets, TrainConfig(steps=2000, batch_size=32768, lr=1e-2, device=device))
        pred = render_full(model, coords, device=device).reshape(H, W, -1).clamp(0, 1)
        table[m][s] = psnr(pred, b)
        if s == 'MetalPlates013': recon[m] = pred
    print('done', s)

done MetalPlates013


done Metal032


done Planks020


done Rock023


## 3. Table 2 — per-material PSNR / 逐材質 PSNR

In [4]:
print(f"{'material':16s} " + ' '.join(f'{m:>10s}' for m in methods))
for s in sets:
    print(f'{s:16s} ' + ' '.join(f'{table[m][s]:10.2f}' for m in methods))
print()
for m in methods:
    print(f'{m:16s} mean PSNR = {np.mean(list(table[m].values())):.2f} dB')

material                ntc  grid_peps   ntc_peps
MetalPlates013        37.79      38.15      38.61
Metal032              52.13      52.13      52.28
Planks020             35.43      36.05      36.28
Rock023               35.13      34.89      35.18

ntc              mean PSNR = 40.12 dB
grid_peps        mean PSNR = 40.31 dB
ntc_peps         mean PSNR = 40.59 dB


## 4. RTXNTC side-by-side on MetalPlates013 (albedo) / 與 RTXNTC 並排(albedo)

In [5]:
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
ax[0].imshow(bundles['MetalPlates013'][..., :3]); ax[0].set_title('target albedo'); ax[0].axis('off')
for i, m in enumerate(methods):
    ax[i+1].imshow(recon[m][..., :3])
    ax[i+1].set_title(f'{m}\n{table[m]["MetalPlates013"]:.1f} dB'); ax[i+1].axis('off')
plt.suptitle('MetalPlates013 — same set NVIDIA RTXNTC demos'); plt.show()

## 5. Save + takeaway / 存檔與小結
PEPS variants lead on metals; the gap narrows on low-frequency wood — the
honest picture the paper's Table 2 also shows.

PEPS 變體在金屬領先;低頻木頭差距縮小 —— 這正是論文 Table 2 的誠實圖像。

In [6]:
import csv
os.makedirs('../results', exist_ok=True)
with open('../results/table2_texture.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['material'] + list(methods))
    for s in sets: w.writerow([s] + [round(table[m][s], 3) for m in methods])
print('saved ../results/table2_texture.csv')

saved ../results/table2_texture.csv
